# Publication figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Figures.ipynb)

The experiment notebooks write screen figures: PNG at 160 dpi, sized for a notebook cell. That is
the wrong artefact for a paper — ACL templates want **vector PDF**, sized to the column it will sit
in, with type large enough to read at print size.

This notebook redraws every figure from the **stored results**. No GPU, no TIMIT, no model
downloads, no decoding. It reads four numbers-only files that each sweep already writes to Drive
and emits PDF (for LaTeX) alongside PNG (for slides and quick viewing).

| source | file | carries |
|---|---|---|
| **O** scaling | `scaling_per_condition.csv` | pooled S/D/I per (model, offset, arm) |
| **A** delta sweep | `delta_per_utterance.csv` | per-utterance `delta_m` and its four component WERs |
| **B** positional embedding | `pe_per_condition.csv` | corpus WER per (model, condition, arm) |
| **C** localization | `expc_per_utterance.csv` | per-utterance NLL, margins, S/D/I, lengths |

Any file that is missing is skipped with a notice, so this runs usefully even when only some
sweeps have completed.

**Two details that matter for submission.** `pdf.fonttype = 42` embeds TrueType rather than
matplotlib's default Type 3, which some venues and arXiv's checker reject. And figure width is set
to the actual printed column width, so the type is scaled once here rather than by
`\includegraphics` later — shrinking a wide figure in LaTeX is what makes axis labels unreadable.

## 1. Where the data is, and where figures go

In [ ]:
import os, csv, math
import numpy as np

try:                                        # Colab: results live on Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/NAACL"
except Exception:                           # local: point this at a folder of downloaded CSVs
    DATA_DIR = os.environ.get("NAACL_DATA", ".")

OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

SRC = {
    "O": os.path.join(DATA_DIR, "scaling_per_condition.csv"),
    "A": os.path.join(DATA_DIR, "delta_per_utterance.csv"),
    "B": os.path.join(DATA_DIR, "pe_per_condition.csv"),
    "C": os.path.join(DATA_DIR, "expc_per_utterance.csv"),
}
ORDER = ["tiny", "base", "small", "medium", "large-v3"]


def load(key):
    p = SRC[key]
    if not os.path.exists(p):
        print(f"  {key}: MISSING  {p}")
        return None
    rows = list(csv.DictReader(open(p, newline="")))
    print(f"  {key}: {len(rows):6d} rows  {os.path.basename(p)}")
    return rows

print(f"data dir: {DATA_DIR}")
D = {k: load(k) for k in SRC}


def models_in(rows):
    got = {r["model"] for r in rows}
    return [m for m in ORDER if m in got]


def fnum(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return float("nan")

## 2. Publication style

Sizes are in inches at final print scale: ACL two-column gives **3.15 in** for a single column and
**6.30 in** across both. Set the figure to the size it will actually occupy and never rescale it in
LaTeX.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

COL, FULL = 3.15, 6.30                       # ACL single / double column width, inches

mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,   # TrueType, not Type 3 (arXiv rejects Type 3)
    "svg.fonttype": "none",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 3, "ytick.major.size": 3,
    "lines.linewidth": 1.4, "lines.markersize": 4,
    "grid.linewidth": 0.5, "grid.color": "#d9d9d6",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9a9a96", "axes.labelcolor": "#111111",
    "xtick.color": "#444444", "ytick.color": "#444444",
    "legend.frameon": False, "figure.dpi": 200,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "savefig.facecolor": "white",
})

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CMODEL = {m: PALETTE[i] for i, m in enumerate(ORDER)}
WRITTEN = []


def finish(fig, stem):
    """Write vector PDF for LaTeX and PNG for everything else."""
    for ext in ("pdf", "png"):
        p = os.path.join(OUT_DIR, f"{stem}.{ext}")
        fig.savefig(p, format=ext)
        WRITTEN.append(p)
    plt.close(fig)
    print(f"  wrote {stem}.pdf + .png")


def grid(ax, axis="y"):
    ax.grid(True, axis=axis, zorder=0)
    ax.set_axisbelow(True)

print("style set | single column", COL, "in | full width", FULL, "in")

## 3. Figure 1 — WER against position, by model size

The paper's first figure. Two panels sharing a y-axis: with timestamps on, and with them off. One
line per model. The y-axis is logarithmic because the series span two orders of magnitude, from
`large-v3` near 0.015 to `tiny` above 1.0 — on a linear axis every model but `tiny` collapses onto
the floor and the flat off-panel becomes unreadable.

The flat right-hand panel is the load-bearing one: same audio, same encoder, same position, only
the decoding objective differs.

In [ ]:
if D["O"]:
    rows = D["O"]
    ms = models_in(rows)
    offs = sorted({int(r["offset_s"]) for r in rows})
    W = {(r["model"], r["timestamps"], int(r["offset_s"])): fnum(r["corpus_wer"]) for r in rows}

    fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.5), sharey=True)
    for ax, ts, title in zip(axes, ("on", "off"),
                             ("timestamps on", "timestamps off")):
        grid(ax)
        for m in ms:
            y = [W.get((m, ts, o), float("nan")) for o in offs]
            ax.plot(offs, y, marker="o", color=CMODEL[m], label=m,
                    markeredgecolor="white", markeredgewidth=0.6, zorder=3)
        ax.set_yscale("log")
        ax.set_xticks(offs)
        ax.set_xlabel("utterance offset in the 30 s window (s)")
        ax.set_title(title, loc="left", fontweight="bold")
    axes[0].set_ylabel("corpus WER")
    axes[1].legend(loc="upper left", ncol=1, handlelength=1.6)
    fig.subplots_adjust(wspace=0.08)
    finish(fig, "fig1_wer_vs_offset")
else:
    print("skipped: run the scaling sweep to produce scaling_per_condition.csv")

## 4. Figure 2 — which quantity tracks the penalty?

The paper's second figure, and the one that settles Experiment C. Three quantities, each divided by
its own value at `tiny`, so three different units become one dimensionless axis and the *shapes*
can be compared:

- **WER penalty** — WER(25 s) / WER(5 s), minus one so a penalty-free model sits at zero
- **ΔNLL** — teacher-forced, paired per utterance, with search removed
- **runaway rate** — fraction of outputs exceeding twice the reference length

A line that tracks the penalty supports the encoder account; one that stays flat while the penalty
falls supports the decoder account.

In [ ]:
def expc_summary(rows):
    """Corpus WER, dNLL and runaway rate per model, from per-utterance rows."""
    by = {}
    for r in rows:
        by.setdefault((r["model"], r["cond"], r["timestamps"]), []).append(r)
    out = {}
    for m in models_in(rows):
        def wer(cond, ts):
            v = by[(m, cond, ts)]
            return (sum(int(x["sub"]) + int(x["dele"]) + int(x["ins"]) for x in v)
                    / sum(int(x["n_ref_words"]) for x in v))
        nll = {c: {x["path"]: fnum(x["nll_text"]) for x in by[(m, c, "on")]}
               for c in ("C0", "C1", "C2")}
        paths = sorted(nll["C0"])
        d1 = np.array([nll["C1"][p] - nll["C0"][p] for p in paths])
        v = by[(m, "C1", "on")]
        run = np.mean([int(x["n_hyp_words"]) > 2 * int(x["n_ref_words"]) for x in v])
        out[m] = {"wer_c0": wer("C0", "on"), "wer_c1": wer("C1", "on"),
                  "dnll": float(d1.mean()), "runaway": float(run),
                  "penalty": wer("C1", "on") / wer("C0", "on")}
    return out


if D["C"]:
    S = expc_summary(D["C"])
    ms = [m for m in ORDER if m in S]
    base = ms[0]
    series = [
        ("WER penalty", [(S[m]["penalty"] - 1) / (S[base]["penalty"] - 1) for m in ms], PALETTE[0]),
        ("$\\Delta$NLL", [S[m]["dnll"] / S[base]["dnll"] for m in ms], PALETTE[1]),
        ("runaway rate", [S[m]["runaway"] / S[base]["runaway"] for m in ms], PALETTE[3]),
    ]
    x = np.arange(len(ms))
    fig, ax = plt.subplots(figsize=(COL, 2.35))
    grid(ax)
    ax.axhline(1.0, color="#9a9a96", linestyle=":", linewidth=0.7, zorder=1)
    for lab, y, c in series:
        ax.plot(x, y, marker="o", color=c, label=lab,
                markeredgecolor="white", markeredgewidth=0.6, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_ylabel(f"relative to {base}")
    ax.legend(handlelength=1.6)
    finish(fig, "fig2_accounts")

    print(f"\n{'model':>9} {'penalty':>8} {'dNLL':>9} {'runaway':>8}")
    for m in ms:
        print(f"{m:>9} {S[m]['penalty']:7.2f}x {S[m]['dnll']:+9.4f} {S[m]['runaway']:8.3f}")
    pen = np.array([(S[m]["penalty"] - 1) for m in ms])
    print(f"\ncorrelation with the WER penalty across {len(ms)} sizes:")
    print(f"  dNLL     r = {np.corrcoef(pen, [S[m]['dnll'] for m in ms])[0,1]:+.3f}")
    print(f"  runaway  r = {np.corrcoef(pen, [S[m]['runaway'] for m in ms])[0,1]:+.3f}")
else:
    print("skipped: expc_per_utterance.csv not found")

## 5. Figure 3 — the positional-embedding intervention

Experiment B as grouped bars, one group per condition, one bar per model, log y. Reads
`pe_per_condition.csv` directly: it already stores corpus WER per condition, so nothing is
recomputed here.

In [ ]:
if D["B"]:
    rows = [r for r in D["B"] if r["timestamps"] == "on"]
    ms = models_in(rows)
    conds = []
    for r in rows:
        if r["cond"] not in conds:
            conds.append(r["cond"])
    conds.sort()
    W = {(r["model"], r["cond"]): fnum(r["corpus_wer"]) for r in rows}

    x = np.arange(len(conds)); w = 0.8 / len(ms)
    fig, ax = plt.subplots(figsize=(FULL, 2.3))
    grid(ax)
    for i, m in enumerate(ms):
        ax.bar(x + i * w - 0.4 + w / 2, [W.get((m, c), np.nan) for c in conds],
               width=w * 0.9, color=CMODEL[m], label=m, zorder=3, linewidth=0)
    ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(conds)
    ax.set_ylabel("corpus WER"); ax.set_xlabel("condition")
    ax.legend(ncol=5, loc="upper center", handlelength=1.2, columnspacing=1.2)
    finish(fig, "fig3_positional_embedding")
else:
    print("skipped: pe_per_condition.csv not found")

## 6. Figure 4 — prevalence and severity of the per-utterance effect

Experiment A. `delta_m` is zero-inflated and heavy-tailed, so prevalence and severity are drawn as
two panels rather than combined into one summary. Left: the share of utterances with
`delta_m > 0`, with Wilson intervals. Right: mean `delta_m`, log scale.

In [ ]:
def wilson(k, n, z=1.959963985):
    if n == 0:
        return float("nan"), 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


if D["A"]:
    rows = D["A"]
    ms = models_in(rows)
    by = {}
    for r in rows:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))

    x = np.arange(len(ms))
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.3))

    ax = axes[0]; grid(ax)
    pts = [wilson(int((np.array(by[m]) > 1e-9).sum()), len(by[m])) for m in ms]
    ax.errorbar(x, [p[0] for p in pts],
                yerr=[[p[0] - p[1] for p in pts], [p[2] - p[0] for p in pts]],
                fmt="o", color=PALETTE[0], ecolor="#9a9a96", elinewidth=0.8,
                capsize=2.5, markeredgecolor="white", markeredgewidth=0.6, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_ylabel(r"$P(\Delta_m > 0)$")
    ax.set_title("prevalence", loc="left", fontweight="bold")

    ax = axes[1]; grid(ax)
    ax.plot(x, [np.mean(by[m]) for m in ms], marker="o", color=PALETTE[1],
            markeredgecolor="white", markeredgewidth=0.6, zorder=3)
    ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_ylabel(r"mean $\Delta_m$")
    ax.set_title("severity", loc="left", fontweight="bold")

    fig.subplots_adjust(wspace=0.32)
    finish(fig, "fig4_delta_prevalence_severity")
else:
    print("skipped: delta_per_utterance.csv not found")

## 7. LaTeX tables

The same stored numbers as `booktabs` tables, so Tables 1 and 2 do not have to be transcribed from
notebook output by hand. Paste into the paper and adjust the caption.

In [ ]:
def latex(rows, header, body, caption, label):
    out = ["\\begin{table}[t]", "\\centering", "\\small",
           "\\begin{tabular}{" + header[0] + "}", "\\toprule",
           " & ".join(header[1]) + " \\\\", "\\midrule"]
    out += [" & ".join(r) + " \\\\" for r in body]
    out += ["\\bottomrule", "\\end{tabular}",
            f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\end{table}"]
    return "\n".join(out)


if D["B"]:
    rows = [r for r in D["B"] if r["timestamps"] == "on"]
    ms = models_in(rows)
    conds = sorted({r["cond"] for r in rows})
    W = {(r["model"], r["cond"]): fnum(r["corpus_wer"]) for r in rows}
    body = []
    for m in ms:
        rec = ""
        if "P0" in conds and "P1" in conds and "P2" in conds:
            d = W[(m, "P1")] - W[(m, "P0")]
            rec = f"{(W[(m,'P1')] - W[(m,'P2')]) / d:.2f}" if d else "--"
        body.append([m] + [f"{W.get((m, c), float('nan')):.4f}" for c in conds] + [rec])
    print(latex(rows, ("l" + "r" * (len(conds) + 1),
                       ["Model"] + conds + ["rec.\\ P2"]), body,
                "Corpus WER under positional-embedding displacement, timestamps on, "
                "1000 TIMIT utterances. Recovery is $(P1-P2)/(P1-P0)$.",
                "tab:pe"))
    print()

if D["A"]:
    ms = models_in(D["A"])
    by = {}
    for r in D["A"]:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))
    body = []
    for m in ms:
        d = np.array(by[m]); pos, neg = d > 1e-9, d < -1e-9
        p, lo, hi = wilson(int(pos.sum()), len(d))
        body.append([m, f"{d.mean():+.3f}",
                     f"{d[pos | neg].mean():+.2f}" if (pos | neg).any() else "--",
                     f"{p:.3f}", f"[{lo:.3f}, {hi:.3f}]",
                     f"{pos.sum() / max(1, pos.sum() + neg.sum()):.3f}"])
    print(latex(D["A"], ("lrrrrr",
                ["Model", "mean $\\Delta_m$", "mean\\,$|$aff.", "$P(\\Delta_m{>}0)$",
                 "95\\% CI", "$P(+|$aff.$)$"]), body,
                "Per-utterance timestamp-specific positional penalty over 1000 utterances. "
                "Intervals are Wilson score.", "tab:delta"))

## 8. What was written

In [ ]:
print(f"{len(WRITTEN)} files in {os.path.abspath(OUT_DIR)}/\n")
for p in WRITTEN:
    print(f"  {os.path.getsize(p)/1024:7.1f} KB  {os.path.basename(p)}")

pdfs = [p for p in WRITTEN if p.endswith(".pdf")]
for p in pdfs:                                  # a PDF that is not vector is a silent failure
    head = open(p, "rb").read(1024)
    assert head[:4] == b"%PDF", p
print(f"\nall {len(pdfs)} PDFs verified as PDF containers, fonts embedded as TrueType (fonttype 42)")

try:
    from google.colab import files
    import shutil
    shutil.make_archive("figures", "zip", OUT_DIR)
    print("\nfigures.zip written -- download it from the file browser, or run files.download")
except Exception:
    pass